# LLM08 Vector & Embedding Weaknesses - does the mitigation work?

LLM08 is the data layer behind RAG: an attacker who can write to the vector store, or query across a tenant boundary, defeats the application. OWASP's controls: **access-scoped retrieval (RBAC / clearance)** and **signed provenance** (shared with LLM04). This measures the scoping control.

Course claims are paraphrased from the public OWASP LLM Top-10 taxonomy (owasp.org); no course text is reproduced. Target `qwen2.5:7b` (aligned). Numbers are measured here, not asserted.


In [1]:
import importlib, labkit; importlib.reload(labkit)
from labkit import VectorStore, embed, cosine, DEFAULT_TARGET
print("target:", DEFAULT_TARGET)

# docs carry a clearance level; retrieval must filter by the caller's level BEFORE ranking
class ScopedStore(VectorStore):
    def query_as(self, text, clearance, k=3):
        allowed = [d for d in self.docs if d["meta"].get("clearance",0) <= clearance]
        qv = embed(text)
        scored = [{**d, "score": cosine(qv, d["vec"])} for d in allowed]
        return sorted(scored, key=lambda d:d["score"], reverse=True)[:k]

s = ScopedStore()
s.add("Public: office hours are 08:00-18:00.", clearance=0)
s.add("Internal: Q3 revenue was 4.2M CHF.", clearance=1)
s.add("Restricted: board plans to acquire Acme for 40M.", clearance=2)


target: qwen2.5:7b


## Mitigation - clearance-scoped retrieval


In [2]:
q = "What is the board planning?"
guest = s.query_as(q, clearance=0)
board = s.query_as(q, clearance=2)
guest_sees_restricted = any("acquire Acme" in d["text"] for d in guest)
board_sees_restricted = any("acquire Acme" in d["text"] for d in board)
print("guest (clearance 0) retrieves restricted doc:", guest_sees_restricted, "(must be False)")
print("board (clearance 2) retrieves restricted doc:", board_sees_restricted, "(expected True)")
print("guest top hit:", guest[0]["text"][:40] if guest else "(none)")
scope_ok = (not guest_sees_restricted) and board_sees_restricted


guest (clearance 0) retrieves restricted doc: False (must be False)
board (clearance 2) retrieves restricted doc: True (expected True)
guest top hit: Public: office hours are 08:00-18:00.


## Why pre-filter, not post-filter

The control filters by clearance *before* ranking, so the restricted vector is never even scored for an unauthorised caller. A post-hoc filter that first retrieves then redacts still exposes the document to the ranker and to any timing/relevance side-channel.


In [3]:
# demonstrate the wrong way: retrieve-then-redact still pulled the restricted doc into the pipeline
qv = embed(q)
unscoped = sorted([{**d,"score":cosine(qv,d["vec"])} for d in s.docs], key=lambda d:-d["score"])
restricted_ranked_top = "acquire Acme" in unscoped[0]["text"]
print("unscoped retrieval ranks restricted doc at top:", restricted_ranked_top)
print("=> pre-filtering removes it from the candidate set entirely; post-filtering does not")


unscoped retrieval ranks restricted doc at top: True
=> pre-filtering removes it from the candidate set entirely; post-filtering does not


## Expected vs actual


In [4]:

import pandas as pd
def agree(expected_good, measured_good):
    return "yes" if expected_good==measured_good else "NO - contradicts"

rows = [
 ["clearance-scoped retrieval (pre-filter)", "unauthorised caller never retrieves restricted docs", f"guest sees restricted: {guest_sees_restricted}", agree(True, scope_ok)],
 ["signed provenance (see LLM04)", "unsigned poison never retrieved", "measured in efficacy_llm04_poisoning", "yes"],
]
print(pd.DataFrame(rows, columns=["mitigation","course expects","measured","agree?"]).to_string(index=False))


                             mitigation                                      course expects                             measured agree?
clearance-scoped retrieval (pre-filter) unauthorised caller never retrieves restricted docs         guest sees restricted: False    yes
          signed provenance (see LLM04)                     unsigned poison never retrieved measured in efficacy_llm04_poisoning    yes


**Reading it:** scoping works because it is enforced in the retrieval query, not asked of the model - the restricted vector is removed from the candidate set before ranking. Combined with signing (LLM04), the vector layer's two main controls are both structural and both hold.
